# Methoden en Technieken 2025-2026 -- Blok 3

## Datapunt Opdracht 3a

In deze opdracht worden de volgende leeruitkomsten getoetst, relevante termen zijn **dik** gedrukt:
- A2: Je stelt voor een AI-oplossing juridische, ethische, organisatorische, **functionele en technische requirements** op.
- B1: Je **verkent en prepareert een dataset voor het trainen en testen van een AI-model en kan de voor- en nadelen van het gebruik van een bestaande dataset onderbouwen**, rekening houdend met technische en ethische randvoorwaarden.
- B2: Je **stelt op basis van requirements en data een geschikte architectuur voor een AI-oplossing op en selecteert daarvoor passende AI-technieken gebruik makend van bijvoorbeeld** **machine learning**, deep learning, kennisrepresentatie, computer vision en **natural language processing**.
- B3: Je **ontwikkelt een nieuw** of voorgetraind **AI-model volgens een iteratief en systematisch proces**.
- C2: **Je evalueert en beoordeelt de kwaliteit van een AI-model aan de hand van kwaliteitscriteria die in het vakgebied erkend worden** zoals robustness, **performance**, scalability, explainability, **model complexity** en resource demand.


## De opdracht

Onderstaande code leest de data van verschillende *ratings* in. Deze dataset is de **MovieTweetings**-dataset (ook naar verwezen in Les 4 van blok 3) waar het MovieGEEKs-voorbeeld gebruik van maakt. In de data staan de waarderingen (van 0 t/m 10) van gebruikers voor verschillende films en bijbehorende *timestamp*. Zie ook https://github.com/sidooms/MovieTweetings/tree/master voor een uitleg van de dataset.

In [7]:
%pip install pandas surprise numpy==1.26.4 scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 8.1/8.1 MB 71.1 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------

In [8]:
import pandas as pd
import numpy as np
import math
import warnings

from surprise import SVD, Dataset, Reader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

In [9]:
ratings = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat',
    delimiter='::', engine='python', header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)
print(f"Loaded {len(ratings):,} ratings")

Loaded 921,398 ratings


In [21]:
items_raw = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/movies.dat',
    delimiter='::', engine='python', header=None,
    names=['movie_id', 'title_raw', 'genres_raw'],
    encoding='utf-8'
)

# Extract year from title, e.g. "Toy Story (1995)" → 1995
items_raw['year'] = items_raw['title_raw'].str.extract(r'\((\d{4})\)').astype(float)
items_raw['title'] = items_raw['title_raw'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True)

# Split genre string into a list (literal '|') and guard against missing values
items_raw['genres_raw'] = items_raw['genres_raw'].fillna('')
items_raw['genres'] = items_raw['genres_raw'].str.split('|', regex=False)
items_raw['genres'] = items_raw['genres'].apply(lambda g: [x for x in g if x])

# Ensure unique movie metadata per movie_id
items = (
    items_raw[['movie_id', 'title', 'year', 'genres']]
    .drop_duplicates(subset=['movie_id'], keep='last')
    .reset_index(drop=True)
)
print(f"Loaded {len(items):,} unique movies")

Loaded 38,013 unique movies


In [11]:
items

,movie_id,title,year,genres
0,8,Edison Kinetoscopic Record of a Sneeze,1894.0,"[Documentary, Short]"
1,10,La sortie des usines Lumière,1895.0,"[Documentary, Short]"
2,12,The Arrival of a Train,1896.0,"[Documentary, Short]"
3,25,The Oxford and Cambridge University Boat Race,1895.0,NaN
4,91,Le manoir du diable,1896.0,"[Short, Horror]"
...,...,...,...,...
38013,15711402,Les rois de l&x27;arnaque,2021.0,"[Crime, Documentary]"
38014,15831978,Cash,2021.0,NaN
38015,15839820,Sompoy,2021.0,"[Comedy, Romance]"
38016,15842076,The Making of &x27;Rocky vs. Drago&x27;,2021.0,[Documentary]


De bedoeling is om een aanbevelings-systeem te bouwen dat voor elke willekeurige gebruiker in het systeem drie films aanbeveelt. Probeer de aanbeveling zo persoonlijk mogelijk te maken.
* Kies een model en verantwoord deze keuze.
* Besluit hoe je het model beoordeelt (datasplitsing en maatstaf) en verantwoord deze keuze.
* Evalueer het model.
* Geef concrete suggesties om het model te verbeteren. Je hoeft deze verbeteringen niet uit te voeren.
* Bespreek voor- en nadelen van het model dat je hebt gemaakt.

In [12]:
ratings

,user_id,movie_id,rating,timestamp
0,1,114508,8,1381006850
1,2,499549,9,1376753198
2,2,1305591,8,1376742507
3,2,1428538,1,1371307089
4,3,75314,1,1595468524
...,...,...,...,...
921393,71705,9893250,10,1613857551
921394,71705,9898858,3,1585958452
921395,71706,172495,10,1587107015
921396,71706,414387,10,1587107852


In [22]:
# ── Merge ratings with movie metadata ─────────────────────────────────────────
df = ratings.merge(items, on='movie_id', how='inner')

# Ensure rating is numeric and within the 0–10 scale
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df[(df['rating'] >= 0) & (df['rating'] <= 10)].dropna(subset=['rating', 'genres'])

print(f"Merged dataset: {len(df):,} ratings  |  rating range {df['rating'].min():.0f}–{df['rating'].max():.0f}")
df.head()

Merged dataset: 921,398 ratings  |  rating range 0–10


,user_id,movie_id,rating,timestamp,title,year,genres
0,1,114508,8,1381006850,Species,1995.0,"[Action, Horror, Sci-Fi, Thriller]"
1,2,499549,9,1376753198,Avatar,2009.0,"[Action, Adventure, Fantasy, Sci-Fi]"
2,2,1305591,8,1376742507,Mars Needs Moms,2011.0,"[Animation, Adventure, Family, Sci-Fi]"
3,2,1428538,1,1371307089,Hansel &amp; Gretel: Witch Hunters,2013.0,"[Action, Fantasy, Horror]"
4,3,75314,1,1595468524,Taxi Driver,1976.0,"[Crime, Drama]"


## 2. Exploratory Analysis

Check dataset size and remove noisy users (fewer than 3 ratings).

In [23]:
# ── Exploratory Checks ────────────────────────────────────────────────────────
n_users  = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()
n_ratings = len(df)
avg_per_user = n_ratings / n_users

print(f"Users:                {n_users:,}")
print(f"Movies:               {n_movies:,}")
print(f"Total ratings:        {n_ratings:,}")
print(f"Avg ratings / user:   {avg_per_user:.2f}")

# Remove users with fewer than 3 ratings to reduce noise
user_counts = df.groupby('user_id').size()
valid_users = user_counts[user_counts >= 3].index
df = df[df['user_id'].isin(valid_users)].copy()

print(f"\nAfter removing users with < 3 ratings:")
print(f"  Users:   {df['user_id'].nunique():,}")
print(f"  Ratings: {len(df):,}")

Users:                71,707
Movies:               38,013
Total ratings:        921,398
Avg ratings / user:   12.85

After removing users with < 3 ratings:
  Users:   31,925
  Ratings: 872,759


## 3. Train / Test Split (time-based, 80 / 20 per user)

For every user the ratings are sorted by timestamp. The first 80 % go into the training set and the remaining 20 % into the test set. This prevents information leakage and respects the chronological order in which a user rated movies.

In [24]:
# ── Time-based train / test split ─────────────────────────────────────────────
def time_based_split(data, train_ratio=0.8):
    """Split each user's ratings chronologically: first 80 % train, last 20 % test."""
    train_parts, test_parts = [], []
    for _, group in data.groupby('user_id'):
        g = group.sort_values('timestamp')
        n = len(g)
        split = max(1, int(n * train_ratio))   # at least 1 in train
        if split >= n:
            split = n - 1                      # at least 1 in test
        train_parts.append(g.iloc[:split])
        test_parts.append(g.iloc[split:])
    return (pd.concat(train_parts).reset_index(drop=True),
            pd.concat(test_parts).reset_index(drop=True))

train_df, test_df = time_based_split(df)

print(f"Train: {len(train_df):,} ratings  ({train_df['user_id'].nunique():,} users)")
print(f"Test:  {len(test_df):,} ratings  ({test_df['user_id'].nunique():,} users)")

Train: 686,166 ratings  (31,925 users)
Test:  186,593 ratings  (31,925 users)


## 4. Baseline — Popularity Recommender

For every movie, compute:
$$\text{popularity\_score} = \overline{r} \cdot \ln(1 + n)$$
where $\overline{r}$ is the average rating and $n$ the number of ratings. This baseline is also used for **cold-start users**.

In [25]:
# ── Baseline: Popularity Recommender ──────────────────────────────────────────
movie_stats = train_df.groupby('movie_id').agg(
    num_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# popularity_score = avg_rating × log(1 + num_ratings)
movie_stats['popularity_score'] = (
    movie_stats['avg_rating'] * np.log1p(movie_stats['num_ratings'])
)
movie_stats = movie_stats.merge(
    items[['movie_id', 'title', 'genres']], on='movie_id', how='left'
).sort_values('popularity_score', ascending=False).reset_index(drop=True)

# Pre-compute fast-lookup dicts
popularity_dict = movie_stats.set_index('movie_id')['popularity_score'].to_dict()
movie_avg_dict  = movie_stats.set_index('movie_id')['avg_rating'].to_dict()

print("Top-10 most popular movies (baseline):\n")
print(
    movie_stats[['title', 'avg_rating', 'num_ratings', 'popularity_score']]
    .head(10)
    .to_string(index=False)
)

Top-10 most popular movies (baseline):

                   title  avg_rating  num_ratings  popularity_score
            Interstellar    8.882114         1968         67.373330
                 Gravity    8.209770         2436         64.024082
 The Wolf of Wall Street    8.336271         2156         63.993164
The Shawshank Redemption    9.329920          879         63.256131
                Whiplash    8.581792         1406         62.211258
                    1917    8.536806         1440         62.088977
           Hacksaw Ridge    8.703883         1236         61.975518
        Django Unchained    8.547690         1342         61.566115
        Captain Phillips    8.209540         1761         61.359784
                   Joker    9.055620          863         61.230233


## 5. Content-Based Filtering (genre similarity)

1. One-hot encode the genre lists into a binary feature matrix.
2. Build a **user genre profile** by averaging the genre vectors of movies rated ≥ 7.
3. Compute **cosine similarity** between each user profile and every movie's genre vector.

In [26]:
# ── Content-Based: genre one-hot matrix + user profiles ───────────────────────

# Ensure genres are always list-like
items['genres'] = items['genres'].apply(
    lambda g: g if isinstance(g, list) else ([] if pd.isna(g) else [str(g)])
)

# One-hot encode genres into a binary movie×genre matrix
mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(
    mlb.fit_transform(items['genres']),
    columns=mlb.classes_,
    index=items['movie_id']
)
# Safety: enforce unique index
genre_matrix = genre_matrix.groupby(level=0).max()

# Drop placeholder genre if it exists
if '(no genres listed)' in genre_matrix.columns:
    genre_matrix.drop(columns=['(no genres listed)'], inplace=True)

all_genre_names = list(genre_matrix.columns)
print(f"Genre features ({len(all_genre_names)}): {', '.join(all_genre_names)}")

# ── Build user genre profiles (avg genre vector of movies rated ≥ 7) ─────────
def build_user_profiles(train_data, genre_mat, threshold=7):
    profiles = {}
    for uid, grp in train_data.groupby('user_id'):
        liked = grp[grp['rating'] >= threshold]['movie_id']
        vecs  = genre_mat.loc[genre_mat.index.intersection(liked)]
        if len(vecs) == 0:                       # fallback: use all rated movies
            vecs = genre_mat.loc[genre_mat.index.intersection(grp['movie_id'])]
        profiles[uid] = vecs.mean().values if len(vecs) > 0 else np.zeros(genre_mat.shape[1])
    return profiles

user_profiles = build_user_profiles(train_df, genre_matrix)

# ── Pre-compute full cosine-similarity matrix (users × movies) ───────────────
user_ids_ordered  = list(user_profiles.keys())
user_profile_mat  = np.array([user_profiles[u] for u in user_ids_ordered])
movie_ids_ordered = genre_matrix.index.tolist()
genre_mat_values  = genre_matrix.values

cs_matrix = cosine_similarity(user_profile_mat, genre_mat_values)   # shape (n_users, n_movies)

user_idx_map  = {u: i for i, u in enumerate(user_ids_ordered)}
movie_idx_map = {m: j for j, m in enumerate(movie_ids_ordered)}

def content_score_lookup(user_id, movie_id):
    """O(1) cosine similarity between a user's genre profile and a movie."""
    ui = user_idx_map.get(user_id)
    mi = movie_idx_map.get(movie_id)
    if ui is None or mi is None:
        return 0.0
    return float(cs_matrix[ui, mi])

print(f"User profiles built for {len(user_profiles):,} users")
print(f"Content-similarity matrix shape: {cs_matrix.shape}")

Genre features (28): Action, Adult, Adventure, Animation, Biography, Comedy, Crime, Documentary, Drama, Family, Fantasy, Film-Noir, Game-Show, History, Horror, Music, Musical, Mystery, News, Reality-TV, Romance, Sci-Fi, Short, Sport, Talk-Show, Thriller, War, Western
User profiles built for 31,925 users
Content-similarity matrix shape: (31925, 38013)


## 6. Collaborative Filtering — SVD (Surprise)

Train a matrix-factorisation model (SVD) on the training ratings using the [Surprise](https://surpriselib.com/) library. The model learns latent user/item factors and predicts ratings for unseen user–movie pairs.

In [27]:
# ── Collaborative Filtering: train SVD ────────────────────────────────────────
reader = Reader(rating_scale=(0, 10))
surprise_data = Dataset.load_from_df(
    train_df[['user_id', 'movie_id', 'rating']], reader
)
trainset = surprise_data.build_full_trainset()

svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

print(f"SVD model trained  —  factors={svd.n_factors}, epochs={svd.n_epochs}")
print(f"Internal trainset: {trainset.n_users} users, {trainset.n_items} items, {trainset.n_ratings} ratings")

SVD model trained  —  factors=100, epochs=20
Internal trainset: 31925 users, 33487 items, 686166 ratings


## 7. Hybrid Scoring Function & Cold-Start Handling

$$\text{final\_score} = 0.7 \times \text{collaborative\_score} + 0.3 \times \text{content\_score}$$

**Cold-start rules:**
- **Users** with < 3 ratings → skip collaborative filtering, use the popularity model with genre diversity.
- **Movies** with < 5 ratings → substitute a neutral collaborative score (global average) so the content signal dominates.

In [29]:
# ── Hybrid scoring with cold-start handling ───────────────────────────────────
COLLAB_WEIGHT  = 0.7
CONTENT_WEIGHT = 0.3

# Pre-compute useful lookups
user_train_counts  = train_df.groupby('user_id').size().to_dict()
movie_train_counts = train_df.groupby('movie_id').size().to_dict()
user_rated_sets    = train_df.groupby('user_id')['movie_id'].apply(set).to_dict()
all_movie_ids      = set(items['movie_id'].unique())
global_avg_rating  = train_df['rating'].mean()

def _batch_svd_predict(user_id, movie_ids):
    """Vectorised SVD predictions using internal factor matrices."""
    n = len(movie_ids)
    try:
        iuid = svd.trainset.to_inner_uid(user_id)
    except ValueError:
        return np.full(n, global_avg_rating)

    pu = svd.pu[iuid]
    bu = svd.bu[iuid]
    mu = svd.trainset.global_mean

    preds  = np.full(n, global_avg_rating)
    valid  = np.zeros(n, dtype=bool)
    iiids  = np.zeros(n, dtype=int)
    for i, mid in enumerate(movie_ids):
        try:
            iiids[i] = svd.trainset.to_inner_iid(mid)
            valid[i] = True
        except ValueError:
            pass                                    # cold movie → keep global avg

    if valid.any():
        qi_batch = svd.qi[iiids[valid]]              # (n_valid, n_factors)
        bi_batch = svd.bi[iiids[valid]]              # (n_valid,)
        preds[valid] = mu + bu + bi_batch + qi_batch @ pu

    return np.clip(preds, 0, 10)

def get_hybrid_score(user_id, movie_id):
    """Single-pair hybrid score (used during evaluation)."""
    is_cold_user = user_train_counts.get(user_id, 0) < 3
    if is_cold_user:
        return popularity_dict.get(movie_id, 0.0)

    is_cold_movie = movie_train_counts.get(movie_id, 0) < 5
    if is_cold_movie:
        collab = global_avg_rating
    else:
        try:
            iuid = svd.trainset.to_inner_uid(user_id)
            iiid = svd.trainset.to_inner_iid(movie_id)
            collab = svd.trainset.global_mean + svd.bu[iuid] + svd.bi[iiid] + np.dot(svd.pu[iuid], svd.qi[iiid])
            collab = np.clip(collab, 0, 10)
        except ValueError:
            collab = global_avg_rating

    cb = content_score_lookup(user_id, movie_id) * 10.0
    return COLLAB_WEIGHT * collab + CONTENT_WEIGHT * cb

print(f"Hybrid weights: collab={COLLAB_WEIGHT}, content={CONTENT_WEIGHT}")
print(f"Cold-start thresholds: user < 3 ratings → popularity | movie < 5 ratings → global avg ({global_avg_rating:.2f})")

Hybrid weights: collab=0.7, content=0.3
Cold-start thresholds: user < 3 ratings → popularity | movie < 5 ratings → global avg (7.26)


## 8. Recommendation Function with Serendipity & Diversity

For each user:
1. Score all unrated movies with the hybrid model.
2. Pick the **top 2** by score.
3. Pick a **diverse 3rd** movie whose genres are *not* a subset of the first two — this adds serendipity.

Cold-start users receive popularity-based picks with enforced genre diversity.

In [30]:
# ── Recommendation generation with diversity / serendipity ────────────────────

def recommend(user_id, n=3):
    """
    Return a DataFrame with the top-n recommended movies for *user_id*.
    Uses the hybrid model for regular users and the popularity model for cold-start users.
    Enforces genre diversity for the third recommendation (serendipity).
    """
    rated      = user_rated_sets.get(user_id, set())
    candidates = np.array(list(all_movie_ids - rated))
    if len(candidates) == 0:
        return pd.DataFrame(columns=['rank', 'movie_id', 'title', 'genres', 'score'])

    is_cold_user = user_train_counts.get(user_id, 0) < 3

    # ── Score every candidate ────────────────────────────────────────────────
    if is_cold_user:
        scores = np.array([popularity_dict.get(m, 0.0) for m in candidates])
    else:
        # Collaborative scores (vectorised)
        collab_scores = _batch_svd_predict(user_id, candidates)
        # Content scores (vectorised)
        ui = user_idx_map.get(user_id)
        if ui is not None:
            cb_scores = np.array([
                cs_matrix[ui, movie_idx_map[m]] * 10.0
                if m in movie_idx_map else 0.0
                for m in candidates
            ])
        else:
            cb_scores = np.zeros(len(candidates))
        scores = COLLAB_WEIGHT * collab_scores + CONTENT_WEIGHT * cb_scores

    # ── Rank and apply diversity rule ────────────────────────────────────────
    order = np.argsort(-scores)
    sorted_cands  = candidates[order]
    sorted_scores = scores[order]

    if len(sorted_cands) <= n or n <= 2:
        return _format_recs(list(zip(sorted_cands[:n], sorted_scores[:n])))

    if is_cold_user:
        return _diverse_popularity_picks(sorted_cands, sorted_scores, n)

    # Top 2 by hybrid score
    top2 = [(sorted_cands[0], sorted_scores[0]),
            (sorted_cands[1], sorted_scores[1])]

    # Genres covered by top 2
    top2_genres = set()
    for mid, _ in top2:
        if mid in movie_idx_map:
            row = genre_matrix.loc[mid]
            top2_genres.update(row[row == 1].index)

    # Diverse 3rd pick: first candidate whose genres are NOT a subset of top-2
    diverse = None
    search_limit = min(80, len(sorted_cands))
    for i in range(2, search_limit):
        mid = sorted_cands[i]
        if mid in movie_idx_map:
            mg = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if mg and not mg.issubset(top2_genres):
                diverse = (mid, sorted_scores[i])
                break
    if diverse is None:
        diverse = (sorted_cands[2], sorted_scores[2])

    return _format_recs([top2[0], top2[1], diverse])


def _diverse_popularity_picks(sorted_cands, sorted_scores, n):
    """Pick the top-n popular movies ensuring each adds at least one new genre."""
    picked = []
    seen_genres = set()
    for mid, sc in zip(sorted_cands, sorted_scores):
        if mid in movie_idx_map:
            mg = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if not mg.issubset(seen_genres) or len(picked) == 0:
                picked.append((mid, sc))
                seen_genres.update(mg)
        if len(picked) >= n:
            break
    # Fill any remaining slots from top
    if len(picked) < n:
        existing = {p[0] for p in picked}
        for mid, sc in zip(sorted_cands, sorted_scores):
            if mid not in existing:
                picked.append((mid, sc))
            if len(picked) >= n:
                break
    return _format_recs(picked[:n])


def _format_recs(picked):
    """Convert list of (movie_id, score) tuples to a presentation DataFrame."""
    rows = []
    for rank, (mid, sc) in enumerate(picked, 1):
        info = items[items['movie_id'] == mid]
        title  = info['title'].values[0] if len(info) else f"Movie {mid}"
        genres = ', '.join(info['genres'].values[0]) if len(info) and isinstance(info['genres'].values[0], list) else ''
        rows.append({
            'rank': rank, 'movie_id': int(mid),
            'title': title, 'genres': genres,
            'score': round(float(sc), 3)
        })
    return pd.DataFrame(rows)


print("recommend() function ready.")

recommend() function ready.


## 9. Evaluation

| Metric | What it measures |
|---|---|
| **RMSE** | Prediction accuracy — how close predicted ratings are to actual test ratings. |
| **Precision@3** | Ranking quality — what fraction of the top-3 recommendations are *relevant* (test rating ≥ 7)? |

Both the **hybrid** model and the **popularity baseline** are compared.

In [31]:
# ── Evaluation: RMSE and Precision@3 ──────────────────────────────────────────

# ---------- 1) RMSE — hybrid model ----------
def compute_rmse_hybrid(test_data):
    uids    = test_data['user_id'].values
    mids    = test_data['movie_id'].values
    actuals = test_data['rating'].values
    preds   = np.empty(len(actuals))

    mu = svd.trainset.global_mean
    for i in range(len(actuals)):
        uid, mid = uids[i], mids[i]
        if user_train_counts.get(uid, 0) < 3:            # cold-start user
            preds[i] = movie_avg_dict.get(mid, global_avg_rating)
            continue
        if movie_train_counts.get(mid, 0) < 5:           # cold-start movie
            collab = global_avg_rating
        else:
            try:
                iuid = svd.trainset.to_inner_uid(uid)
                iiid = svd.trainset.to_inner_iid(mid)
                collab = mu + svd.bu[iuid] + svd.bi[iiid] + np.dot(svd.pu[iuid], svd.qi[iiid])
                collab = np.clip(collab, 0, 10)
            except ValueError:
                collab = global_avg_rating
        cb = content_score_lookup(uid, mid) * 10.0
        preds[i] = COLLAB_WEIGHT * collab + CONTENT_WEIGHT * cb

    preds = np.clip(preds, 0, 10)
    return math.sqrt(mean_squared_error(actuals, preds))

# ---------- 2) RMSE — popularity baseline ----------
def compute_rmse_baseline(test_data):
    actuals = test_data['rating'].values
    preds   = np.array([movie_avg_dict.get(m, global_avg_rating) for m in test_data['movie_id'].values])
    return math.sqrt(mean_squared_error(actuals, preds))

# ---------- 3) Precision@K ----------
def precision_at_k(test_data, rec_func, k=3, threshold=7):
    precisions = []
    for uid in test_data['user_id'].unique():
        relevant = set(test_data[(test_data['user_id'] == uid) & (test_data['rating'] >= threshold)]['movie_id'])
        if not relevant:
            continue                                      # skip users with no relevant test items
        recs = rec_func(uid, n=k)
        if recs.empty:
            precisions.append(0.0)
            continue
        hits = len(set(recs['movie_id']) & relevant)
        precisions.append(hits / k)
    return np.mean(precisions) if precisions else 0.0

# ---------- 4) Baseline recommend function ----------
def baseline_recommend(user_id, n=3):
    rated = user_rated_sets.get(user_id, set())
    top = movie_stats[~movie_stats['movie_id'].isin(rated)].head(n)
    rows = []
    for rank, (_, r) in enumerate(top.iterrows(), 1):
        g = ', '.join(r['genres']) if isinstance(r['genres'], list) else str(r['genres'])
        rows.append({'rank': rank, 'movie_id': r['movie_id'],
                     'title': r['title'], 'genres': g,
                     'score': round(r['popularity_score'], 3)})
    return pd.DataFrame(rows)

# ---------- Run evaluation ----------
print("Computing RMSE on the full test set …")
rmse_hybrid   = compute_rmse_hybrid(test_df)
rmse_baseline = compute_rmse_baseline(test_df)

# Precision@3 on a sample of test users (full sweep is slow)
sample_size    = min(300, test_df['user_id'].nunique())
sample_users   = np.random.choice(test_df['user_id'].unique(), size=sample_size, replace=False)
sample_test_df = test_df[test_df['user_id'].isin(sample_users)]

print(f"Computing Precision@3 on {sample_size} sampled test users …")
p3_hybrid   = precision_at_k(sample_test_df, recommend, k=3, threshold=7)
p3_baseline = precision_at_k(sample_test_df, baseline_recommend, k=3, threshold=7)

# ---------- Print results ----------
print(f"\n{'Metric':<20} {'Hybrid':>10} {'Baseline':>10}")
print(f"{'-'*42}")
print(f"{'RMSE':<20} {rmse_hybrid:>10.4f} {rmse_baseline:>10.4f}")
print(f"{'Precision@3':<20} {p3_hybrid:>10.4f} {p3_baseline:>10.4f}")

Computing RMSE on the full test set …
Computing Precision@3 on 300 sampled test users …

Metric                   Hybrid   Baseline
------------------------------------------
RMSE                     1.7482     1.6372
Precision@3              0.0073     0.0256


## 10. Example Recommendations

Show the top-3 recommendations for several users, including their predicted hybrid score and the genres of each recommended movie.

In [33]:
# ── Example recommendations for several users ────────────────────────────────
# Pick 5 users with varying activity levels
active_users = train_df['user_id'].value_counts()
example_users = list(active_users.index[:3])          # 3 most active users

# Also include a less-active user (fewest ratings among valid users)
least_active = active_users.tail(2).index.tolist()
example_users.extend(least_active)

for uid in example_users:
    n_rated = user_train_counts.get(uid, 0)
    tag = "  ← cold-start" if n_rated < 3 else ""
    print(f"\n{'='*72}")
    print(f"  User {uid}   ({n_rated} training ratings){tag}")
    print(f"{'='*72}")
    recs = recommend(uid, n=3)
    for _, row in recs.iterrows():
        print(f"  #{int(row['rank'])}  {row['title']:<50s}  score={row['score']:.3f}")
        print(f"      Genres: {row['genres']}")


  User 17405   (2300 training ratings)
  #1  12 Angry Men                                        score=8.803
      Genres: Crime, Drama
  #2  Gravity                                             score=8.718
      Genres: Drama, Sci-Fi, Thriller
  #3  Inglourious Basterds                                score=8.706
      Genres: Adventure, Drama, War

  User 26962   (1656 training ratings)
  #1  Mulholland Dr.                                      score=9.226
      Genres: Drama, Mystery, Thriller
  #2  The Dark Knight                                     score=9.178
      Genres: Action, Crime, Drama, Thriller
  #3  Donnie Darko                                        score=8.741
      Genres: Drama, Sci-Fi, Thriller

  User 40861   (1539 training ratings)
  #1  Joker                                               score=8.769
      Genres: Crime, Drama, Thriller
  #2  The Lives of Others                                 score=8.753
      Genres: Drama, Thriller
  #3  The Dark Knight         